In [4]:
import mlflow
import mlflow.sklearn
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (f1_score, precision_score, 
                            recall_score, accuracy_score)
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Apuntar al mlruns de la raíz del proyecto
mlflow.set_tracking_uri('../../mlruns')
mlflow.set_experiment("hate_speech_detection")
print(mlflow.get_tracking_uri())

nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\r\n|\r|\n', ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words
        and len(t) > 2
    ]
    return ' '.join(tokens)

# Cargar datos
df = pd.read_csv('../../data/processed/comments_processed.csv')
X = df['text_clean']
y = df['IsToxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Configurar MLflow
mlflow.set_experiment("hate_speech_detection")
print("MLflow configurado ✅")



c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Traceback (most recent call last):
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "c:\Users\zula

../../mlruns


Traceback (most recent call last):
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\ml

MLflow configurado ✅


In [5]:
# Función helper para loggear un experimento
def log_experiment(run_name, model, tfidf, params, X_train_vec, X_test_vec, y_train, y_test):
    with mlflow.start_run(run_name=run_name):
        # Log parámetros
        mlflow.log_params(params)

        # Entrenar
        model.fit(X_train_vec, y_train)

        # Métricas train
        y_train_pred = model.predict(X_train_vec)
        train_f1 = f1_score(y_train, y_train_pred, average='weighted')

        # Métricas test
        y_test_pred = model.predict(X_test_vec)
        test_f1 = f1_score(y_test, y_test_pred, average='weighted')
        test_precision = precision_score(y_test, y_test_pred, average='weighted')
        test_recall = recall_score(y_test, y_test_pred, average='weighted')
        test_accuracy = accuracy_score(y_test, y_test_pred)
        gap = abs(train_f1 - test_f1)

        # Log métricas
        mlflow.log_metrics({
            'train_f1': train_f1,
            'test_f1': test_f1,
            'test_precision': test_precision,
            'test_recall': test_recall,
            'test_accuracy': test_accuracy,
            'overfitting_gap': gap
        })

        # Log modelo
        mlflow.sklearn.log_model(model, "model")

        print(f"✅ {run_name} — Test F1: {test_f1:.3f} | Gap: {gap:.3f}")

# --- Experimento 1: Baseline Logistic Regression ---
tfidf_1 = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
X_train_1 = tfidf_1.fit_transform(X_train)
X_test_1 = tfidf_1.transform(X_test)

log_experiment(
    run_name="baseline_logistic_regression",
    model=LogisticRegression(random_state=42, max_iter=1000),
    tfidf=tfidf_1,
    params={'model': 'LogisticRegression', 'tfidf_features': 5000, 
            'ngram_range': '(1,2)', 'min_df': 2, 'C': 1.0},
    X_train_vec=X_train_1, X_test_vec=X_test_1,
    y_train=y_train, y_test=y_test
)

# --- Experimento 2: Ensemble regularizado ---
tfidf_2 = TfidfVectorizer(max_features=2000, ngram_range=(1,2), min_df=3)
X_train_2 = tfidf_2.fit_transform(X_train)
X_test_2 = tfidf_2.transform(X_test)

ensemble_reg = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(C=0.1, random_state=42, max_iter=1000)),
        ('svm', SVC(C=0.1, kernel='linear', probability=True, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                            subsample=0.8, random_state=42, eval_metric='logloss'))
    ], voting='soft'
)

log_experiment(
    run_name="ensemble_regularized",
    model=ensemble_reg,
    tfidf=tfidf_2,
    params={'model': 'VotingEnsemble', 'tfidf_features': 2000,
            'ngram_range': '(1,2)', 'min_df': 3,
            'lr_C': 0.1, 'svm_C': 0.1, 'xgb_depth': 3},
    X_train_vec=X_train_2, X_test_vec=X_test_2,
    y_train=y_train, y_test=y_test
)

# --- Experimento 3: Ensemble Optuna v2 (modelo final) ---
tfidf_final = TfidfVectorizer(max_features=500, ngram_range=(1,2), min_df=5)
X_train_final = tfidf_final.fit_transform(X_train)
X_test_final = tfidf_final.transform(X_test)

ensemble_final = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(C=0.2463, random_state=42, max_iter=1000)),
        ('svm', SVC(C=0.0184, kernel='linear', probability=True, random_state=42)),
        ('xgb', XGBClassifier(max_depth=2, learning_rate=0.0592, n_estimators=150,
                            subsample=0.686, random_state=42, eval_metric='logloss'))
    ], voting='soft'
)

log_experiment(
    run_name="ensemble_optuna_v2_FINAL",
    model=ensemble_final,
    tfidf=tfidf_final,
    params={'model': 'VotingEnsemble_Optuna', 'tfidf_features': 500,
            'ngram_range': '(1,2)', 'min_df': 5,
            'lr_C': 0.2463, 'svm_C': 0.0184,
            'xgb_depth': 2, 'xgb_lr': 0.0592,
            'xgb_n': 150, 'subsample': 0.686},
    X_train_vec=X_train_final, X_test_vec=X_test_final,
    y_train=y_train, y_test=y_test
)

print("\nTodos los experimentos registrados ✅")

2026/05/21 12:30:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 12:30:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/21 12:31:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


✅ baseline_logistic_regression — Test F1: 0.750 | Gap: 0.188


2026/05/21 12:31:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 12:31:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/21 12:31:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


✅ ensemble_regularized — Test F1: 0.720 | Gap: 0.147


2026/05/21 12:31:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 12:31:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/21 12:31:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


✅ ensemble_optuna_v2_FINAL — Test F1: 0.719 | Gap: 0.100

Todos los experimentos registrados ✅


In [6]:
import scipy.sparse as sp
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import joblib
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (f1_score, precision_score,
                            recall_score, accuracy_score)
from xgboost import XGBClassifier

mlflow.set_experiment("hate_speech_detection")

# Cargar matrices del dataset enriquecido
X_train_enr = sp.load_npz('../../data/vectorized/X_train_tfidf.npz')
X_test_enr  = sp.load_npz('../../data/vectorized/X_test_tfidf.npz')
y_train_enr = pd.read_csv('../../data/vectorized/y_train.csv').squeeze()
y_test_enr  = pd.read_csv('../../data/vectorized/y_test.csv').squeeze()

print(f"Matrices cargadas: {X_train_enr.shape} | {X_test_enr.shape}")


from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Función helper adaptada para matrices ya vectorizadas
def log_experiment_enr(run_name, model, params, 
                        X_train_vec, X_test_vec, 
                        y_train, y_test, dataset='enriched_10k'):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({**params, 'dataset': dataset})
        model.fit(X_train_vec, y_train)
        
        y_train_pred = model.predict(X_train_vec)
        y_test_pred  = model.predict(X_test_vec)
        
        train_f1  = f1_score(y_train, y_train_pred, average='weighted')
        test_f1   = f1_score(y_test,  y_test_pred,  average='weighted')
        test_prec = precision_score(y_test, y_test_pred, average='weighted')
        test_rec  = recall_score(y_test,  y_test_pred,  average='weighted')
        test_acc  = accuracy_score(y_test, y_test_pred)
        gap       = abs(train_f1 - test_f1)
        
        mlflow.log_metrics({
            'train_f1':        train_f1,
            'test_f1':         test_f1,
            'test_precision':  test_prec,
            'test_recall':     test_rec,
            'test_accuracy':   test_acc,
            'overfitting_gap': gap
        })
        
        mlflow.sklearn.log_model(model, "model")
        print(f"✅ {run_name} — Test F1: {test_f1:.3f} | Gap: {gap:.3f} {'✅' if gap < 0.05 else '⚠️'}")

# --- Modelos dataset enriquecido ---
log_experiment_enr(
    run_name="enr_logreg_baseline",
    model=LogisticRegression(max_iter=1000, random_state=42),
    params={'model': 'LogisticRegression', 'C': 1.0, 
            'tfidf_features': 5000, 'ngram_range': '(1,2)'},
    X_train_vec=X_train_enr, X_test_vec=X_test_enr,
    y_train=y_train_enr, y_test=y_test_enr
)

log_experiment_enr(
    run_name="enr_logreg_C01",
    model=LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    params={'model': 'LogisticRegression', 'C': 0.1,
            'tfidf_features': 5000, 'ngram_range': '(1,2)'},
    X_train_vec=X_train_enr, X_test_vec=X_test_enr,
    y_train=y_train_enr, y_test=y_test_enr
)

# LinearSVC necesita CalibratedClassifierCV para predict_proba
svc_calibrated = CalibratedClassifierCV(
    LinearSVC(max_iter=2000, random_state=42)
)
log_experiment_enr(
    run_name="enr_linearsvc",
    model=svc_calibrated,
    params={'model': 'LinearSVC', 'max_iter': 2000,
            'tfidf_features': 5000, 'ngram_range': '(1,2)'},
    X_train_vec=X_train_enr, X_test_vec=X_test_enr,
    y_train=y_train_enr, y_test=y_test_enr
)

log_experiment_enr(
    run_name="enr_random_forest",
    model=RandomForestClassifier(n_estimators=100, random_state=42),
    params={'model': 'RandomForest', 'n_estimators': 100,
            'tfidf_features': 5000, 'ngram_range': '(1,2)'},
    X_train_vec=X_train_enr, X_test_vec=X_test_enr,
    y_train=y_train_enr, y_test=y_test_enr
)

log_experiment_enr(
    run_name="enr_random_forest_v2",
    model=RandomForestClassifier(n_estimators=100, max_depth=10,
                                min_samples_leaf=5, max_features='sqrt',
                                random_state=42),
    params={'model': 'RandomForest_v2', 'n_estimators': 100,
            'max_depth': 10, 'min_samples_leaf': 5,
            'tfidf_features': 5000, 'ngram_range': '(1,2)'},
    X_train_vec=X_train_enr, X_test_vec=X_test_enr,
    y_train=y_train_enr, y_test=y_test_enr
)

log_experiment_enr(
    run_name="enr_logreg_balanced_FINAL",
    model=LogisticRegression(C=1.0, max_iter=1000,
                            class_weight='balanced', random_state=42),
    params={'model': 'LogisticRegression_balanced', 'C': 1.0,
            'class_weight': 'balanced', 'tfidf_features': 5000,
            'ngram_range': '(1,2)'},
    X_train_vec=X_train_enr, X_test_vec=X_test_enr,
    y_train=y_train_enr, y_test=y_test_enr
)

print("\nTodos los experimentos del dataset enriquecido registrados ✅")

Traceback (most recent call last):
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
  File "c:\Users\zulay\Desktop\FactoriaF5\Bootcamp\IA\projects\P9E4\.venv\Lib\site-packages\ml

Matrices cargadas: (8331, 5000) | (2083, 5000)


2026/05/21 12:31:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 12:31:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/21 12:31:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/21 12:31:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ enr_logreg_baseline — Test F1: 0.910 | Gap: 0.038 ✅


2026/05/21 12:31:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/21 12:31:31 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

✅ enr_random_forest — Test F1: 0.901 | Gap: 0.096 ⚠️


2026/05/21 12:32:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 12:32:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/21 12:32:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


✅ enr_random_forest_v2 — Test F1: 0.827 | Gap: 0.003 ✅


2026/05/21 12:32:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 12:32:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/21 12:32:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


✅ enr_logreg_balanced_FINAL — Test F1: 0.909 | Gap: 0.039 ✅

Todos los experimentos del dataset enriquecido registrados ✅


## Experimentos registrados en MLflow

| Run | Dataset | Test F1 | Gap |
|---|---|---|---|
| baseline_logreg | 1k | 0.750 | 0.188 ⚠️ |
| ensemble_regularized | 1k | 0.748 | 0.102 ⚠️ |
| ensemble_optuna_v2 | 1k | 0.757 | **0.035 ✅** |
| enr_logreg_baseline | 10k | 0.910 | 0.038 ✅ |
| enr_logreg_C01 | 10k | 0.896 | **0.009 ✅** |
| enr_linearsvc | 10k | 0.913 | 0.070 ⚠️ |
| enr_random_forest | 10k | 0.901 | 0.096 ⚠️ |
| enr_random_forest_v2 | 10k | 0.827 | 0.003 ✅ |
| **enr_logreg_balanced_FINAL** | **10k** | **0.909** | **0.039 ✅** |

Los warnings de MLflow sobre serialización en pickle son esperados y no afectan
al funcionamiento. Se recomienda migrar a formato `skops` en versiones futuras.

Modelos que cumplen el requisito de overfitting < 5%:
- `enr_logreg_C01` — mejor gap (0.009) pero F1 inferior (0.896)
- `enr_logreg_baseline` — F1 0.910, gap 0.038
- `enr_logreg_balanced_FINAL` — **modelo seleccionado**, F1 0.909, gap 0.039
- `ensemble_optuna_v2` — mejor modelo con dataset original (1k)